# CDS Curve Stripping — Simple vs Exact Iterative Model


In [ ]:
## Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq  # For root finding

Q1) Simple Model - continuous premium payments

In [ ]:
maturities = np.array((1, 3, 5, 7, 10))
R = np.array((100,110,120,120,125))/10000 # Convert bps to decimal
LGD = 0.4
r = 0.03

## Average Hazard Rate

lambda_avg = R / LGD

## Cumulative Hazard Rate and default Probability

H = lambda_avg * maturities
Q = np.exp(-H)
P_Default = 1 - Q

## Forward hazard rate
H_prev = np.r_[0, H[:-1]]
T_prev = np.r_[0, maturities[:-1]]
lambda_forward = (H - H_prev) / (maturities - T_prev)


## Default Probablity between periods
Q_prev = np.r_[1, Q[:-1]]
P_Default_period = Q_prev - Q

## Output table
output_table = pd.DataFrame({
    'Maturity (in years)': maturities,
    'CDS Rate (in bps)': R*10000,
    'Average Hazard Rate': lambda_avg,
    'Forward Hazard Rate': lambda_forward,
    'Default Probability Period': P_Default_period
})

print(output_table.to_string(index=False))


In [ ]:
# --- Visualization of the Term Structure ---
plt.figure(figsize=(10, 6))

# 1. Plot Average Hazard Rates (Point Estimates)
# These represent the average intensity from t=0 to t=T
plt.plot(maturities, lambda_avg, 'o--', color='navy', alpha=0.7, 
         label=r'Average Hazard Rate $\bar{\lambda}(0, T)$')

# 2. Plot Forward Hazard Rates (Buckets)
# These represent the constant intensity specific to the interval (T_prev, T]
# We use hlines to explicitly show the "width" of the time bucket.
for t_start, t_end, rate in zip(T_prev, maturities, lambda_forward):
    plt.hlines(y=rate, xmin=t_start, xmax=t_end, colors='darkorange', linewidth=2.5)
    # Optional: Add vertical dotted lines to show bucket boundaries
    plt.vlines(x=t_end, ymin=min(lambda_avg), ymax=max(lambda_forward), 
               colors='gray', linestyles=':', alpha=0.3)

# Dummy plot for legend (to avoid repeating label for every hline)
plt.plot([], [], color='darkorange', linewidth=2.5, label=r'Forward Hazard Rate $\lambda_{fwd}(T_{i-1}, T_i)$')

# Formatting
plt.title(r'Term Structure of Credit Risk: Average vs. Forward Hazard Rates', fontsize=14)
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Hazard Rate ($\lambda$)', fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(np.arange(0, 11, 1))

plt.tight_layout()
plt.show()

Result make sense, CDS Rate increasing, hazard rate is increasing and default porobability as well which is very intuitive. One key observation here is of 7Y maturity where we have same CDS rate as 5Y so hazard rate is same and as a result default probablity decreased slightly compared to previous bucket. 

Q2) Exact Model (Iterative Striping) - Quaterly Premium payments

In [ ]:
## Quarterly schedule --> CDS premium become quarterly payments instead of continuous

def schedule_quarters(T,dt =0.25):
    n = int(round(T/dt))
    return np.array([i*dt for i in range(1,n+1)],dtype=float)

In [ ]:
## cumulative hazard function for piecewise constant hazard rates

def cumulative_hazard(t,nodes,lambdas):

    prev = 0.0
    H = 0.0

    for end, lam in zip(nodes,lambdas):
        if t <= prev:
            break
        seg_end = min(t,end)
        if seg_end > prev:
            H += lam * (seg_end - prev)
            prev = seg_end
        if t <= end:
            break
    return H

## Survival probablity Q(T) = exp(-H(T)) where H(T) is the cumulative hazard function at time T

def Q_survival(t, nodes, lambdas):
    return np.exp(-cumulative_hazard(t, nodes, lambdas))

In [ ]:
## CDS PV = PV(premium leg) - PV(protection leg)
## Premium leg has two components --> Regular premium + accrued premium

def cds_value(T,R,r,LGD,nodes,lambdas, dt=0.25):
    pay_times = schedule_quarters(T, dt)

    pv_prem_reg = 0.0
    pv_prem_acc = 0.0
    pv_prot = 0.0

    T_prev = 0.0

    for Ti in pay_times:
        dT   = Ti - T_prev             
        Tmid = 0.5 * (Ti + T_prev)    

    
        Q_prev = Q_survival(T_prev, nodes, lambdas)  
        Q_i    = Q_survival(Ti,    nodes, lambdas)   

     
        dDefault = Q_prev - Q_i

     
        pv_prem_reg += np.exp(-r * Ti) * dT * Q_i

    
        pv_prem_acc += np.exp(-r * Tmid) * dDefault * (dT / 2.0) # Midpoint approximation for accrued premium

    
        pv_prot += np.exp(-r * Tmid) * dDefault

        T_prev = Ti

    PV_prem = R * (pv_prem_reg + pv_prem_acc)
    PV_prot = LGD * pv_prot

    return PV_prem - PV_prot


In [ ]:
## Bootstrapping --> We will solve for λ1..λ5 such that each quoted CDS is at par (CDS PV = 0)
## Doing this iteratively, solving for one λ at a time

nodes = np.array([1,3,5,7,10], dtype=float)
spreads_bps = np.array([100,110,120,120,125], dtype=float)

def strip_hazards_exact(nodes, spreads_bps, r, LGD, dt=0.25):

    nodes = np.array(nodes, dtype=float)
    spreads_bps = np.array(spreads_bps, dtype=float)

    lambdas_solved = []

    for k, T in enumerate(nodes):
        Rk = spreads_bps[k] / 10000.0  # convert to decimal

        
        def objective(lam_k):
           
            lam_vec = np.array(lambdas_solved + [lam_k], dtype=float)

            if len(lam_vec) < len(nodes):
                lam_vec = np.r_[lam_vec, np.zeros(len(nodes) - len(lam_vec))]

            return cds_value(T, Rk, r, LGD, nodes, lam_vec, dt=dt)

        
        lo, hi = 1e-10, 5.0
        f_lo, f_hi = objective(lo), objective(hi)

       
        while f_lo * f_hi > 0:
            hi *= 2.0
            if hi > 200:
                raise RuntimeError(f"Could not bracket root for maturity {T}Y")
            f_hi = objective(hi)

        lam_star = brentq(objective, lo, hi)
        lambdas_solved.append(lam_star)

        print(f"Solved λ{k+1} for bucket ending at {T}Y: {lam_star:.6f}")

    return np.array(lambdas_solved, dtype=float)


lambdas = strip_hazards_exact(nodes, spreads_bps, r=0.03, LGD=0.40, dt=0.25)
print("Forward hazard curve:", lambdas)



Q3 --> Model validation

In [ ]:
# PV of premium and protection leg for 7Y CDS and find their difference
T = 7.0
R7 = R[3]          # 7Y spread in decimal
dt = 0.25

pay_times = schedule_quarters(T, dt)

pv_prem_reg = 0.0
pv_prem_acc = 0.0
pv_prot = 0.0
T_prev = 0.0

for Ti in pay_times:
    dT = Ti - T_prev
    Tmid = 0.5 * (Ti + T_prev)

    Q_prev = Q_survival(T_prev, nodes, lambdas)
    Q_i    = Q_survival(Ti,    nodes, lambdas)

    dDefault = Q_prev - Q_i

    pv_prem_reg += np.exp(-r * Ti)   * dT * Q_i
    pv_prem_acc += np.exp(-r * Tmid) * dDefault * (dT / 2.0)
    pv_prot     += np.exp(-r * Tmid) * dDefault

    T_prev = Ti

PV_premium = R7 * (pv_prem_reg + pv_prem_acc)
PV_protection = LGD * pv_prot
diff = PV_premium - PV_protection

print(f"PV Premium Leg (7Y):     {PV_premium:.10f}")
print(f"PV Protection Leg (7Y):  {PV_protection:.10f}")
print(f"Difference:              {diff:.10f}")
print("Within 1e-6?", abs(diff) < 1e-6)


Q4 --> Need to change the r from 0.03 to 0.1 and rerun the code again to see the difference